In [ ]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
INPUT_DIR = 'data'
!ls {INPUT_DIR}

In [ ]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'timeout', 'done': True, 'execution_count': None}
import numpy as np
import pandas as pd

rating_df = pd.read_csv(INPUT_DIR + '/rating_complete.csv', 
                        low_memory=False, 
                        usecols=["user_id", "anime_id", "rating"]
                        )
rating_df.head(4)

In [ ]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
n_ratings = rating_df['user_id'].value_counts()
rating_df = rating_df[rating_df['user_id'].isin(n_ratings[n_ratings >= 400].index)].copy()
len(rating_df)

In [ ]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
# Scaling BTW (0 , 1.0)
min_rating = min(rating_df['rating'])
max_rating = max(rating_df['rating'])
rating_df['rating'] = rating_df["rating"].apply(lambda x: (x - min_rating) / (max_rating - min_rating)).values.astype(np.float64)

AvgRating = np.mean(rating_df['rating'])
print('Avg', AvgRating)

In [ ]:
# --- [CELL 4]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
# === BEFORE (original) ===
# # Encoding categorical data
# user_ids = rating_df["user_id"].unique().tolist()[:1000]
# user2user_encoded = {x: i for i, x in enumerate(user_ids)}
# user_encoded2user = {i: x for i, x in enumerate(user_ids)}
# rating_df["user"] = rating_df["user_id"].map(user2user_encoded)
# n_users = len(user2user_encoded)
# 
# anime_ids = rating_df["anime_id"].unique().tolist()[:1000]
# anime2anime_encoded = {x: i for i, x in enumerate(anime_ids)}
# anime_encoded2anime = {i: x for i, x in enumerate(anime_ids)}
# rating_df["anime"] = rating_df["anime_id"].map(anime2anime_encoded)
# n_animes = len(anime2anime_encoded)
# 
# print("Num of users: {}, Num of animes: {}".format(n_users, n_animes))
# print("Min rating: {}, Max rating: {}".format(min(rating_df['rating']), max(rating_df['rating'])))

# === AFTER (edited) ===
# Create encodings based on the sampled data (which is now limited to 1000 rows)
user_ids = rating_df["user_id"].unique().tolist()
user2user_encoded = {x: i for i, x in enumerate(user_ids)}
user_encoded2user = {i: x for i, x in enumerate(user_ids)}
rating_df["user"] = rating_df["user_id"].map(user2user_encoded)
n_users = len(user2user_encoded)

anime_ids = rating_df["anime_id"].unique().tolist()
anime2anime_encoded = {x: i for i, x in enumerate(anime_ids)}
anime_encoded2anime = {i: x for i, x in enumerate(anime_ids)}
rating_df["anime"] = rating_df["anime_id"].map(anime2anime_encoded)
n_animes = len(anime2anime_encoded)

# Define X and y after encoding
X = rating_df[['user', 'anime']].values
y = rating_df["rating"]

print("Num of users: {}, Num of animes: {}".format(n_users, n_animes))
print("Min rating: {}, Max rating: {}".format(min(rating_df['rating']), max(rating_df['rating'])))

In [ ]:
# --- [CELL 5]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
# === BEFORE (original) ===
# # Shuffle
# rating_df = rating_df.sample(frac=1, random_state=73)
# 
# rating_df= rating_df.head(1000)
# 
# X = rating_df[['user', 'anime']].values
# y = rating_df["rating"]

# === AFTER (edited) ===
# Sample and define X, y first (moved from original position)
rating_df = rating_df.sample(frac=1, random_state=73)
rating_df = rating_df.head(1000)

In [ ]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
rating_df.shape[0]

In [ ]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
# Split
# test_set_size = 200 #200 for test set
# train_indices = rating_df.shape[0] - test_set_size 

from sklearn.model_selection import train_test_split

# Limit to 1000 rows
limit_rows = 1000

# Split into 80:20 ratio
test_set_size = int(0.2 * rating_df.shape[0])  # 20% of the limited dataset for the test set
train_indices = rating_df.shape[0] - test_set_size 

# X_train, X_test, y_train, y_test = (
#     X[:train_indices],
#     X[train_indices:],
#     y[:train_indices],
#     y[train_indices:],
# )

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



print('> Train set ratings: {}'.format(len(y_train)))
print('> Test set ratings: {}'.format(len(y_test)))
print(len(X_train))
print(len(X_test))

In [ ]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
X_train_array = [X_train[:, 0], X_train[:, 1]]
X_test_array = [X_test[:, 0], X_test[:, 1]]

In [ ]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [ ]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
# Embedding layers
from tensorflow.keras.layers import Add, Activation, Lambda, BatchNormalization, Concatenate, Dropout, Input, Embedding, Dot, Reshape, Dense, Flatten

def RecommenderNet():
    embedding_size = 128
    
    user = Input(name = 'user', shape = [1])
    user_embedding = Embedding(name = 'user_embedding',
                       input_dim = n_users, 
                       output_dim = embedding_size)(user)
    
    anime = Input(name = 'anime', shape = [1])
    anime_embedding = Embedding(name = 'anime_embedding',
                       input_dim = n_animes, 
                       output_dim = embedding_size)(anime)
    
    #x = Concatenate()([user_embedding, anime_embedding])
    x = Dot(name = 'dot_product', normalize = True, axes = 2)([user_embedding, anime_embedding])
    x = Flatten()(x)
        
    x = Dense(1, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation("sigmoid")(x)
    
    model = Model(inputs=[user, anime], outputs=x)
    model.compile(loss='binary_crossentropy', metrics=["mae", "mse"], optimizer='adam')
    
    return model

model = RecommenderNet()

model.summary()

In [ ]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'aborted', 'done': True, 'execution_count': None}
# Callbacks
from tensorflow.keras.callbacks import Callback, ModelCheckpoint, LearningRateScheduler, TensorBoard, EarlyStopping, ReduceLROnPlateau

start_lr = 0.00001
min_lr = 0.00001
max_lr = 0.00005
batch_size = 100

# if TPU_INIT:
#     max_lr = max_lr * tpu_strategy.num_replicas_in_sync
#     batch_size = batch_size * tpu_strategy.num_replicas_in_sync

rampup_epochs = 5
sustain_epochs = 0
exp_decay = .8

def lrfn(epoch):
    if epoch < rampup_epochs:
        return (max_lr - start_lr)/rampup_epochs * epoch + start_lr
    elif epoch < rampup_epochs + sustain_epochs:
        return max_lr
    else:
        return (max_lr - min_lr) * exp_decay**(epoch-rampup_epochs-sustain_epochs) + min_lr


lr_callback = LearningRateScheduler(lambda epoch: lrfn(epoch), verbose=0)

checkpoint_filepath = 'weights.weights.h5'

model_checkpoints = ModelCheckpoint(filepath=checkpoint_filepath,
                                        save_weights_only=True,
                                        monitor='val_loss',
                                        mode='min',
                                        save_best_only=True)

early_stopping = EarlyStopping(patience = 3, monitor='val_loss', 
                               mode='min', restore_best_weights=True)

my_callbacks = [
    model_checkpoints,
    lr_callback,
    early_stopping,   
]

In [ ]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'timeout', 'done': True, 'execution_count': None}
# Model training
history = model.fit(
    x=X_train_array,
    y=y_train,
    batch_size=batch_size,
    epochs=5,
    verbose=1,
    validation_data=(X_test_array, y_test),
    callbacks=my_callbacks
)

In [ ]:
import numpy as np

# Safely convert X_train_array
if isinstance(X_train_array, list):
    X_train_combined = np.concatenate([np.array(x).reshape(-1, 1) for x in X_train_array], axis=1)
else:
    X_train_combined = np.array(X_train_array)

# Validate no NaN values
assert np.isnan(X_train_combined).sum() == 0, \
    "X_train_array contains NaN values"

# Validate data integrity
assert X_train_combined.shape[0] > 0, "X_train_array has no samples"
assert X_train_combined.shape[1] == 2, "X_train_array should have 2 features"

# CRITICAL: Encoding vocabulary must be complete (no [:1000] slice)
# Fixed version: n_users >> 1000, n_animes >> 1000
# Buggy version: n_users ≈ 1000, n_animes ≈ 1000
assert n_users > 1000 and n_animes > 1000, \
    f"Insufficient encoding vocabulary (n_users={n_users}, n_animes={n_animes}) - " \
    "fix requires removing [:1000] slice from unique IDs"

# Verify encoded values fit within vocabulary
assert np.max(X_train_combined) < max(n_users, n_animes), \
    "Encoded values exceed vocabulary size"